In [1]:
USE {
	dependencies("org.duckdb:duckdb_jdbc:1.5.5.1")
}

In [2]:
%use dataframe
// %use lets-plot
// %use kandy
// %use plotly

import kotlinx.serialization.json.Json

In [3]:
import org.jetbrains.kotlinx.dataframe.io.DbConnectionConfig
import org.jetbrains.kotlinx.dataframe.schema.DataFrameSchema

In [4]:
val db = "/Users/darronporter/Downloads/Dissertation/code/output/results/gcloud_baseline_results_2026-08-31/baseline_results.duckdb"
val url = "jdbc:duckdb:$db"
val username = "duckdb"
val password = "password"

val dbConfig = DbConnectionConfig(url, username, password)

val tableName = "game_results"

In [26]:
val mctsDb = "/Users/darronporter/Downloads/Dissertation/code/output/results/gcloud_baseline_results_2026-08-31/mcts_mcts/mcts_mcts.db"
val mctsUrl = "jdbc:duckdb:$mctsDb"

val mctsDBConfig = DbConnectionConfig(mctsUrl, username, password)

In [27]:
val miniDb = "/Users/darronporter/Downloads/Dissertation/code/output/results/gcloud_baseline_results_2026-08-31/mini_mini/mini_mini.duckdb"
val miniUrl = "jdbc:duckdb:$miniDb"

val miniDBConfig = DbConnectionConfig(miniUrl, username, password)

In [7]:
val winner = DataFrame.readSqlQuery(
	dbConfig,
	sqlQuery = "SELECT seed, turn, winner.winCondition, winner.winner, winner.loser FROM game_results",
)

In [50]:
DataFrame.readSqlQuery(
	dbConfig,
	sqlQuery = """
WITH games AS (
    SELECT
        winner.winner.model      AS winner_model,
        winner.winner.depth      AS winner_depth,
        winner.winner.iterations AS winner_iterations,
        winner.winner.name       AS winner_colour,

        winner.loser.model       AS loser_model,
        winner.loser.depth       AS loser_depth,
        winner.loser.iterations  AS loser_iterations,
        winner.loser.name        AS loser_colour
    FROM game_results
),

normalized AS (
    SELECT
        CASE WHEN winner_model = 'MCTS' THEN winner_iterations ELSE loser_iterations END AS mcts_iter,
        CASE WHEN winner_model = 'MINIMAX' THEN winner_depth ELSE loser_depth END AS mm_depth,

        CASE WHEN winner_colour = 'BLACK' THEN winner_model ELSE loser_model END AS black_model,
        winner_model,
        winner_colour
    FROM games
)

SELECT
    mcts_iter AS "MCTS Iter",
    mm_depth  AS "MM Depth",

    COUNT(*) FILTER (WHERE black_model = 'MCTS' AND winner_model = 'MINIMAX') AS "MM (W) vs MCTS (B) MM Win",
    COUNT(*) FILTER (WHERE black_model = 'MCTS' AND winner_model = 'MCTS')    AS "MM (W) vs MCTS (B) MCTS Win",
    COUNT(*) FILTER (WHERE black_model = 'MCTS')    AS "MM (W) vs MCTS (B) Games",
    ROUND(
        COUNT(*) FILTER (WHERE black_model = 'MCTS' AND winner_model = 'MINIMAX') * 100.0 /
        NULLIF(COUNT(*) FILTER (WHERE black_model = 'MCTS'), 0), 4
    ) || '%' AS "MM (W) Win %",
    ROUND(
        COUNT(*) FILTER (WHERE black_model = 'MCTS' AND winner_model = 'MCTS') * 100.0 /
        NULLIF(COUNT(*) FILTER (WHERE black_model = 'MCTS'), 0), 4
    ) || '%' AS "MCTS (B) Win %",


    COUNT(*) FILTER (WHERE black_model = 'MINIMAX' AND winner_model = 'MCTS')    AS "MCTS (W) vs MM (B) MCTS Win",
    COUNT(*) FILTER (WHERE black_model = 'MINIMAX' AND winner_model = 'MINIMAX') AS "MCTS (W) vs MM (B) MM Win",
    COUNT(*) FILTER (WHERE black_model = 'MINIMAX') AS "MCTS (W) vs MM (B) GAMES",
    ROUND(
        COUNT(*) FILTER (WHERE black_model = 'MINIMAX' AND winner_model = 'MCTS') * 100.0 /
        NULLIF(COUNT(*) FILTER (WHERE black_model = 'MINIMAX'), 0), 4
    ) || '%' AS "MCTS (W) Win %",

    ROUND(
        COUNT(*) FILTER (WHERE black_model = 'MINIMAX' AND winner_model = 'MINIMAX') * 100.0 /
        NULLIF(COUNT(*) FILTER (WHERE black_model = 'MINIMAX'), 0), 4
    ) || '%' AS "MM (B) Win %",

    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MCTS') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "Combined MCTS Win %",

    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MINIMAX') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "Combined MM Win %",
  COUNT(*) AS "Total Games"
FROM normalized
GROUP BY mcts_iter, mm_depth
ORDER BY mcts_iter ASC, mm_depth ASC
    """.trimIndent()
)

MCTS Iter,MM Depth,MM (W) vs MCTS (B) MM Win,MM (W) vs MCTS (B) MCTS Win,MM (W) vs MCTS (B) Games,MM (W) Win %,MCTS (B) Win %,MCTS (W) vs MM (B) MCTS Win,MCTS (W) vs MM (B) MM Win,MCTS (W) vs MM (B) GAMES,MCTS (W) Win %,MM (B) Win %,Combined MCTS Win %,Combined MM Win %,Total Games
0,1,473,27,500,94.6%,5.4%,34,466,500,6.8%,93.2%,6.1%,93.9%,1000
0,3,413,87,500,82.6%,17.4%,65,435,500,13.0%,87.0%,15.2%,84.8%,1000
999,1,157,343,500,31.4%,68.6%,381,119,500,76.2%,23.8%,72.4%,27.6%,1000
999,3,239,261,500,47.8%,52.2%,266,234,500,53.2%,46.8%,52.7%,47.3%,1000
null,null,0,0,0,null,null,0,0,0,null,null,0.0%,0.0%,5


In [49]:
DataFrame.readSqlQuery(
	mctsDBConfig,
	sqlQuery = """
WITH games AS (
   SELECT
       winner.winner.model      AS winner_model,
       winner.winner.iterations AS winner_iterations,
       winner.winner.name       AS winner_colour,

       winner.loser.model       AS loser_model,
       winner.loser.iterations  AS loser_iterations,
       winner.loser.name        AS loser_colour
   FROM game_results
   WHERE winner.winner.model = 'MCTS' AND winner.loser.model = 'MCTS'
),

normalized AS (
   SELECT
       winner_iterations,
       winner_colour,
       CASE WHEN winner_colour = 'BLACK' THEN winner_iterations ELSE loser_iterations END AS black_iterations
   FROM games
   WHERE (winner_iterations = 0 AND loser_iterations = 999)
      OR (winner_iterations = 999 AND loser_iterations = 0)
)

SELECT
   COUNT(*) FILTER (WHERE black_iterations = 0 AND winner_iterations = 0)   AS "MCTS i=999 (W) vs MCTS i=0 (B) i=0 Win",
   COUNT(*) FILTER (WHERE black_iterations = 0 AND winner_iterations = 999) AS "MCTS i=999 (W) vs MCTS i=0 (B) i=999 Win",
   COUNT(*) FILTER (WHERE black_iterations = 0) AS "MCTS i=999 (W) vs MCTS i=0 (B) Games",

   ROUND(
       COUNT(*) FILTER (WHERE black_iterations = 0 AND winner_iterations = 0) * 100.0 /
       NULLIF(COUNT(*) FILTER (WHERE black_iterations = 0), 0), 4
   ) || '%' AS "MCTS i=0 (B) Win %",

      ROUND(
       COUNT(*) FILTER (WHERE black_iterations = 0 AND winner_iterations = 999) * 100.0 /
       NULLIF(COUNT(*) FILTER (WHERE black_iterations = 0), 0), 4
   ) || '%' AS "MCTS i=999 (W) Win %",

   COUNT(*) FILTER (WHERE black_iterations = 999 AND winner_iterations = 999) AS "MCTS i=0 (W) vs MCTS i=999 (B) i=999 Win",
   COUNT(*) FILTER (WHERE black_iterations = 999 AND winner_iterations = 0)   AS "MCTS i=0 (W) vs MCTS i=999 (B) i=0 Win",
   COUNT(*) FILTER (WHERE black_iterations = 999)   AS "MCTS i=0 (W) vs MCTS i=999 (B) Games",
   ROUND(
       COUNT(*) FILTER (WHERE black_iterations = 999 AND winner_iterations = 999) * 100.0 /
       NULLIF(COUNT(*) FILTER (WHERE black_iterations = 999), 0), 4
   ) || '%' AS "MCTS i=999 (B) Win %",

  ROUND(
       COUNT(*) FILTER (WHERE black_iterations = 999 AND winner_iterations = 0) * 100.0 /
       NULLIF(COUNT(*) FILTER (WHERE black_iterations = 999), 0), 4
   ) || '%' AS "MCTS i=0 (W) Win %",

   ROUND(
       COUNT(*) FILTER (WHERE winner_iterations = 0) * 100.0 /
       COUNT(*), 4
   ) || '%' AS "Combined MCTS i=0 Win %",

   ROUND(
       COUNT(*) FILTER (WHERE winner_iterations = 999) * 100.0 /
       COUNT(*), 4
   ) || '%' AS "Combined MCTS i=999 Win %",

   COUNT(*) AS "Total Games"
FROM normalized
    """.trimIndent()
)

MCTS i=999 (W) vs MCTS i=0 (B) i=0 Win,MCTS i=999 (W) vs MCTS i=0 (B) i=999 Win,MCTS i=999 (W) vs MCTS i=0 (B) Games,MCTS i=0 (B) Win %,MCTS i=999 (W) Win %,MCTS i=0 (W) vs MCTS i=999 (B) i=999 Win,MCTS i=0 (W) vs MCTS i=999 (B) i=0 Win,MCTS i=0 (W) vs MCTS i=999 (B) Games,MCTS i=999 (B) Win %,MCTS i=0 (W) Win %,Combined MCTS i=0 Win %,Combined MCTS i=999 Win %,Total Games
0,500,500,0.0%,100.0%,500,0,500,100.0%,0.0%,0.0%,100.0%,1000


In [47]:
DataFrame.readSqlQuery(
	miniDBConfig,
	sqlQuery = """
WITH games AS (
   SELECT
       winner.winner.model      AS winner_model,
       winner.winner.depth      AS winner_depth,
       winner.winner.name       AS winner_colour,

       winner.loser.model       AS loser_model,
       winner.loser.depth       AS loser_depth,
       winner.loser.name        AS loser_colour
   FROM game_results
   WHERE winner.winner.model = 'MINIMAX' AND winner.loser.model = 'MINIMAX'
),

normalized AS (
   SELECT
       winner_depth,
       winner_colour,
       -- Identify which agent played as BLACK
       CASE WHEN winner_colour = 'BLACK' THEN winner_depth ELSE loser_depth END AS black_depth
   FROM games
   WHERE (winner_depth = 1 AND loser_depth = 3)
      OR (winner_depth = 3 AND loser_depth = 1)
)

SELECT
   -- Orientation 1: MM d=1 (Black) vs MM d=3 (White)
   COUNT(*) FILTER (WHERE black_depth = 1 AND winner_depth = 1) AS "MM d=1 (B) vs MM d=3 (W) d=1 Win",
   COUNT(*) FILTER (WHERE black_depth = 1 AND winner_depth = 3) AS "MM d=1 (B) vs MM d=3 (W) d=3 Win",
   COUNT(*) FILTER (WHERE black_depth = 1) AS "MM d=1 (B) vs MM d=3 (W) Games",
   ROUND(
       COUNT(*) FILTER (WHERE black_depth = 1 AND winner_depth = 1) * 100.0 /
       NULLIF(COUNT(*) FILTER (WHERE black_depth = 1), 0), 4
   ) || '%' AS "MM d=1 (B) Win %",
   ROUND(
       COUNT(*) FILTER (WHERE black_depth = 1 AND winner_depth = 3) * 100.0 /
       NULLIF(COUNT(*) FILTER (WHERE black_depth = 1), 0), 4
   ) || '%' AS "MM d=3 (W) Win %",

   -- Orientation 2: MM d=3 (Black) vs MM d=1 (White)
   COUNT(*) FILTER (WHERE black_depth = 3 AND winner_depth = 3) AS "MM d=3 (B) vs MM d=1 (W) d=3 Win",
   COUNT(*) FILTER (WHERE black_depth = 3 AND winner_depth = 1) AS "MM d=3 (B) vs MM d=1 (W) d=1 Win",
   COUNT(*) FILTER (WHERE black_depth = 3) AS "MM d=3 (B) vs MM d=1 (W) Games",
   ROUND(
       COUNT(*) FILTER (WHERE black_depth = 3 AND winner_depth = 3) * 100.0 /
       NULLIF(COUNT(*) FILTER (WHERE black_depth = 3), 0), 4
   ) || '%' AS "MM d=3 (B) Win %",
  ROUND(
       COUNT(*) FILTER (WHERE black_depth = 3 AND winner_depth = 1) * 100.0 /
       NULLIF(COUNT(*) FILTER (WHERE black_depth = 3), 0), 4
   ) || '%' AS "MM d=1 (W) Win %",

   -- Combined Totals
   ROUND(
       COUNT(*) FILTER (WHERE winner_depth = 1) * 100.0 /
       COUNT(*), 4
   ) || '%' AS "Combined MM d=1 Win %",

   ROUND(
       COUNT(*) FILTER (WHERE winner_depth = 3) * 100.0 /
       COUNT(*), 4
   ) || '%' AS "Combined MM d=3 Win %",

   COUNT(*) AS "Total Games"
FROM normalized
""".trimIndent()
)

MM d=1 (B) vs MM d=3 (W) d=1 Win,MM d=1 (B) vs MM d=3 (W) d=3 Win,MM d=1 (B) vs MM d=3 (W) Games,MM d=1 (B) Win %,MM d=3 (W) Win %,MM d=3 (B) vs MM d=1 (W) d=3 Win,MM d=3 (B) vs MM d=1 (W) d=1 Win,MM d=3 (B) vs MM d=1 (W) Games,MM d=3 (B) Win %,MM d=1 (W) Win %,Combined MM d=1 Win %,Combined MM d=3 Win %,Total Games
159,341,500,31.8%,68.2%,341,159,500,68.2%,31.8%,31.8%,68.2%,1000


In [7]:
DataFrame.readSqlQuery(
	dbConfig,
	sqlQuery = "SELECT gameState FROM game_results LIMIT 1",
)

gameState,,,,
currentPlayer,nextPlayer,bitboard,turnMoves,turnDuration
"{""name""=WHITE, model=MCTS, strength=N...","{""name""=BLACK, model=MINIMAX, strengt...","{whiteDVONNLayer=[134222080, 4096, 0,...","{""1""= ...","{""1""=[3], ""2""=[252], ""3""=[0], ""4""=[12..."


In [5]:
DataFrame.readSqlQuery(
	dbConfig,
	sqlQuery = """
WITH turns AS (
    SELECT
        CAST(t.key AS INTEGER) AS turn_number,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN 'WHITE'
            ELSE 'BLACK'
        END AS player,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'WHITE'
                        THEN g.gameState.currentPlayer.model
                    ELSE g.gameState.nextPlayer.model
                END
            ELSE
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'BLACK'
                        THEN g.gameState.currentPlayer.model
                    ELSE g.gameState.nextPlayer.model
                END
        END AS model,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'WHITE'
                        THEN g.gameState.currentPlayer.depth
                    ELSE g.gameState.nextPlayer.depth
                END
            ELSE
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'BLACK'
                        THEN g.gameState.currentPlayer.depth
                    ELSE g.gameState.nextPlayer.depth
                END
        END AS depth,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'WHITE'
                        THEN g.gameState.currentPlayer.iterations
                    ELSE g.gameState.nextPlayer.iterations
                END
            ELSE
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'BLACK'
                        THEN g.gameState.currentPlayer.iterations
                    ELSE g.gameState.nextPlayer.iterations
                END
        END AS iterations,

        (
            SELECT sum(CAST(d.value AS DOUBLE))
            FROM json_each(t.value) AS d
        ) AS turn_duration

    FROM game_results AS g,
         json_each(to_json(g.gameState.turnDuration)) AS t
)

SELECT
    player,
    model,
    depth,
    iterations,

    count(*) AS turns,

    round(sum(turn_duration), 2)        AS total_duration,
    round(avg(turn_duration), 2)        AS mean_turn_duration,
    round(median(turn_duration), 2)     AS median_turn_duration,
    round(stddev_samp(turn_duration), 2) AS stddev_turn_duration,

    round(min(turn_duration), 2) AS min_turn_duration,
    round(quantile_cont(turn_duration, 0.25), 2) AS p25_turn_duration,
    round(quantile_cont(turn_duration, 0.75), 2) AS p75_turn_duration,
    round(quantile_cont(turn_duration, 0.90), 2) AS p90_turn_duration,
    round(quantile_cont(turn_duration, 0.95), 2) AS p95_turn_duration,
    round(quantile_cont(turn_duration, 0.99), 2) AS p99_turn_duration,
    round(max(turn_duration), 2) AS max_turn_duration

FROM turns

GROUP BY
    player,
    model,
    depth,
    iterations

ORDER BY
    player,
    model,
    depth,
    iterations
""".trimIndent(),
)

kotlin-logging: initializing... active logger factory: Slf4jLoggerFactory


player,model,depth,iterations,turns,total_duration,mean_turn_duration,median_turn_duration,stddev_turn_duration,min_turn_duration,p25_turn_duration,p75_turn_duration,p90_turn_duration,p95_turn_duration,p99_turn_duration,max_turn_duration
BLACK,MCTS,1,0,91000,82.000000,0.000000,0.000000,0.090000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000
BLACK,MCTS,1,999,91000,18786433.000000,568.010000,555.000000,382.780000,2.000000,231.000000,857.000000,989.000000,1077.350000,1590.000000,3702.000000
BLACK,MINIMAX,1,0,91000,639480.000000,22.070000,3.000000,85.570000,0.000000,1.000000,8.000000,31.000000,98.000000,420.220000,3874.000000
BLACK,MINIMAX,3,0,91000,55729411.000000,1716.970000,164.000000,7217.000000,0.000000,43.000000,991.000000,3578.000000,7052.900000,25031.870000,337267.000000
WHITE,MCTS,1,0,91000,4871.000000,0.180000,0.000000,5.050000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,152.000000
WHITE,MCTS,1,999,91000,19785677.000000,580.510000,571.000000,383.810000,3.000000,240.000000,875.500000,998.000000,1092.000000,1602.360000,3457.000000
WHITE,MINIMAX,1,0,91000,621600.000000,20.800000,3.000000,78.690000,0.000000,1.000000,8.000000,31.000000,93.000000,368.000000,2811.000000
WHITE,MINIMAX,3,0,91000,52338362.000000,1642.760000,164.000000,8168.550000,0.000000,42.000000,976.000000,3362.200000,6702.100000,22753.980000,700226.000000


In [25]:
DataFrame.readSqlQuery(
	dbConfig,
	sqlQuery = """
WITH turns AS (
    SELECT
        CAST(t.key AS INTEGER) AS turn_number,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN 'WHITE'
            ELSE 'BLACK'
        END AS player,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'WHITE'
                        THEN g.gameState.currentPlayer.model
                    ELSE g.gameState.nextPlayer.model
                END
            ELSE
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'BLACK'
                        THEN g.gameState.currentPlayer.model
                    ELSE g.gameState.nextPlayer.model
                END
        END AS model,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'WHITE'
                        THEN g.gameState.currentPlayer.depth
                    ELSE g.gameState.nextPlayer.depth
                END
            ELSE
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'BLACK'
                        THEN g.gameState.currentPlayer.depth
                    ELSE g.gameState.nextPlayer.depth
                END
        END AS depth,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'WHITE'
                        THEN g.gameState.currentPlayer.iterations
                    ELSE g.gameState.nextPlayer.iterations
                END
            ELSE
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'BLACK'
                        THEN g.gameState.currentPlayer.iterations
                    ELSE g.gameState.nextPlayer.iterations
                END
        END AS iterations,

        (
            SELECT sum(CAST(d.value AS DOUBLE))
            FROM json_each(t.value) AS d
        ) AS turn_duration

    FROM game_results AS g,
         json_each(to_json(g.gameState.turnDuration)) AS t
)

SELECT
    model,
    depth,
    iterations,

    CASE WHEN model = 'MINIMAX' THEN 'MINIMAX (d = ' || depth || ')' ELSE 'MCTS' || ' (i = ' || iterations || ')' END AS config,

    count(*) AS turns,

    round(sum(turn_duration), 2)        AS total,
    round(avg(turn_duration), 2)        AS mean,
    round(median(turn_duration), 2)     AS median,
    round(stddev_samp(turn_duration), 2) AS stddev,

    round(min(turn_duration), 2) AS min,
    round(quantile_cont(turn_duration, 0.25), 2) AS p25,
    round(quantile_cont(turn_duration, 0.75), 2) AS p75,
    round(quantile_cont(turn_duration, 0.90), 2) AS p90,
    round(quantile_cont(turn_duration, 0.95), 2) AS p95,
    round(quantile_cont(turn_duration, 0.99), 2) AS p99,
    round(max(turn_duration), 2) AS max

FROM turns

GROUP BY
    model,
    depth,
    iterations

ORDER BY
    model,
    depth,
    iterations
""".trimIndent(),
)

model,depth,iterations,config,turns,total,mean,median,stddev,min,p25,p75,p90,p95,p99,max
MCTS,1,0,MCTS (i = 0),182000,4953.000000,0.090000,0.000000,3.600000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,152.000000
MCTS,1,999,MCTS (i = 999),182000,38572110.000000,574.360000,563.000000,383.350000,2.000000,235.000000,867.000000,994.000000,1085.000000,1596.000000,3702.000000
MINIMAX,1,0,MINIMAX (d = 1),182000,1261080.000000,21.420000,3.000000,82.150000,0.000000,1.000000,8.000000,31.000000,95.000000,388.000000,3874.000000
MINIMAX,3,0,MINIMAX (d = 3),182000,108067773.000000,1680.210000,164.000000,7703.080000,0.000000,42.000000,985.000000,3480.300000,6863.000000,23850.150000,700226.000000


In [28]:
DataFrame.readSqlQuery(
	miniDBConfig,
	sqlQuery = """
WITH turns AS (
    SELECT
        CAST(t.key AS INTEGER) AS turn_number,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN 'WHITE'
            ELSE 'BLACK'
        END AS player,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'WHITE'
                        THEN g.gameState.currentPlayer.model
                    ELSE g.gameState.nextPlayer.model
                END
            ELSE
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'BLACK'
                        THEN g.gameState.currentPlayer.model
                    ELSE g.gameState.nextPlayer.model
                END
        END AS model,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'WHITE'
                        THEN g.gameState.currentPlayer.depth
                    ELSE g.gameState.nextPlayer.depth
                END
            ELSE
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'BLACK'
                        THEN g.gameState.currentPlayer.depth
                    ELSE g.gameState.nextPlayer.depth
                END
        END AS depth,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'WHITE'
                        THEN g.gameState.currentPlayer.iterations
                    ELSE g.gameState.nextPlayer.iterations
                END
            ELSE
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'BLACK'
                        THEN g.gameState.currentPlayer.iterations
                    ELSE g.gameState.nextPlayer.iterations
                END
        END AS iterations,

        (
            SELECT sum(CAST(d.value AS DOUBLE))
            FROM json_each(t.value) AS d
        ) AS turn_duration

    FROM game_results AS g,
         json_each(to_json(g.gameState.turnDuration)) AS t
)

SELECT
    model,
    depth,
    iterations,

    CASE WHEN model = 'MINIMAX' THEN 'MINIMAX (d = ' || depth || ')' ELSE 'MCTS' || ' (i = ' || iterations || ')' END AS config,

    count(*) AS turns,

    round(sum(turn_duration), 2)        AS total,
    round(avg(turn_duration), 2)        AS mean,
    round(median(turn_duration), 2)     AS median,
    round(stddev_samp(turn_duration), 2) AS stddev,

    round(min(turn_duration), 2) AS min,
    round(quantile_cont(turn_duration, 0.25), 2) AS p25,
    round(quantile_cont(turn_duration, 0.75), 2) AS p75,
    round(quantile_cont(turn_duration, 0.90), 2) AS p90,
    round(quantile_cont(turn_duration, 0.95), 2) AS p95,
    round(quantile_cont(turn_duration, 0.99), 2) AS p99,
    round(max(turn_duration), 2) AS max

FROM turns

GROUP BY
    model,
    depth,
    iterations

ORDER BY
    model,
    depth,
    iterations
""".trimIndent(),
)

model,depth,iterations,config,turns,total,mean,median,stddev,min,p25,p75,p90,p95,p99,max
MINIMAX,1,0,MINIMAX (d = 1),57000,198262.000000,8.020000,2.000000,29.580000,0.000000,1.000000,4.000000,17.000000,31.000000,112.000000,1614.000000
MINIMAX,3,0,MINIMAX (d = 3),57000,52700672.000000,2094.290000,104.000000,10241.700000,0.000000,31.000000,600.250000,3621.700000,9636.650000,35920.280000,383979.000000


In [29]:
DataFrame.readSqlQuery(
	mctsDBConfig,
	sqlQuery = """
WITH turns AS (
    SELECT
        CAST(t.key AS INTEGER) AS turn_number,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN 'WHITE'
            ELSE 'BLACK'
        END AS player,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'WHITE'
                        THEN g.gameState.currentPlayer.model
                    ELSE g.gameState.nextPlayer.model
                END
            ELSE
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'BLACK'
                        THEN g.gameState.currentPlayer.model
                    ELSE g.gameState.nextPlayer.model
                END
        END AS model,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'WHITE'
                        THEN g.gameState.currentPlayer.depth
                    ELSE g.gameState.nextPlayer.depth
                END
            ELSE
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'BLACK'
                        THEN g.gameState.currentPlayer.depth
                    ELSE g.gameState.nextPlayer.depth
                END
        END AS depth,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'WHITE'
                        THEN g.gameState.currentPlayer.iterations
                    ELSE g.gameState.nextPlayer.iterations
                END
            ELSE
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'BLACK'
                        THEN g.gameState.currentPlayer.iterations
                    ELSE g.gameState.nextPlayer.iterations
                END
        END AS iterations,

        (
            SELECT sum(CAST(d.value AS DOUBLE))
            FROM json_each(t.value) AS d
        ) AS turn_duration

    FROM game_results AS g,
         json_each(to_json(g.gameState.turnDuration)) AS t
)

SELECT
    model,
    depth,
    iterations,

    CASE WHEN model = 'MINIMAX' THEN 'MINIMAX (d = ' || depth || ')' ELSE 'MCTS' || ' (i = ' || iterations || ')' END AS config,

    count(*) AS turns,

    round(sum(turn_duration), 2)        AS total,
    round(avg(turn_duration), 2)        AS mean,
    round(median(turn_duration), 2)     AS median,
    round(stddev_samp(turn_duration), 2) AS stddev,

    round(min(turn_duration), 2) AS min,
    round(quantile_cont(turn_duration, 0.25), 2) AS p25,
    round(quantile_cont(turn_duration, 0.75), 2) AS p75,
    round(quantile_cont(turn_duration, 0.90), 2) AS p90,
    round(quantile_cont(turn_duration, 0.95), 2) AS p95,
    round(quantile_cont(turn_duration, 0.99), 2) AS p99,
    round(max(turn_duration), 2) AS max

FROM turns

GROUP BY
    model,
    depth,
    iterations

ORDER BY
    model,
    depth,
    iterations
""".trimIndent(),
)

model,depth,iterations,config,turns,total,mean,median,stddev,min,p25,p75,p90,p95,p99,max
MCTS,1,0,MCTS (i = 0),48500,4622.000000,0.170000,0.000000,4.840000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,145.000000
MCTS,1,999,MCTS (i = 999),48500,19679348.000000,694.970000,739.000000,396.730000,2.000000,396.000000,947.000000,1058.000000,1269.200000,1703.000000,4314.000000


In [6]:
val duration_stats = DataFrame.readSqlQuery(
	dbConfig,
	sqlQuery = """
WITH turns AS (
    SELECT
        CAST(t.key AS INTEGER) AS turn_number,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN 'WHITE'
            ELSE 'BLACK'
        END AS player,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'WHITE'
                        THEN g.gameState.currentPlayer.model
                    ELSE g.gameState.nextPlayer.model
                END
            ELSE
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'BLACK'
                        THEN g.gameState.currentPlayer.model
                    ELSE g.gameState.nextPlayer.model
                END
        END AS model,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'WHITE'
                        THEN g.gameState.currentPlayer.depth
                    ELSE g.gameState.nextPlayer.depth
                END
            ELSE
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'BLACK'
                        THEN g.gameState.currentPlayer.depth
                    ELSE g.gameState.nextPlayer.depth
                END
        END AS depth,

        CASE
            WHEN CAST(t.key AS INTEGER) % 2 = 1 THEN
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'WHITE'
                        THEN g.gameState.currentPlayer.iterations
                    ELSE g.gameState.nextPlayer.iterations
                END
            ELSE
                CASE
                    WHEN upper(g.gameState.currentPlayer.name) = 'BLACK'
                        THEN g.gameState.currentPlayer.iterations
                    ELSE g.gameState.nextPlayer.iterations
                END
        END AS iterations,

        (
            SELECT sum(CAST(d.value AS DOUBLE))
            FROM json_each(t.value) AS d
        ) AS turn_duration

    FROM game_results AS g,
         json_each(to_json(g.gameState.turnDuration)) AS t
)

SELECT
    player,
    model,
    depth,
    iterations,

    count(*) AS turns,

    round(sum(turn_duration), 2)        AS total_duration,
    round(avg(turn_duration), 2)        AS mean_turn_duration,
    round(median(turn_duration), 2)     AS median_turn_duration,
    round(stddev_samp(turn_duration), 2) AS stddev_turn_duration,

    round(min(turn_duration), 2) AS min_turn_duration,
    round(quantile_cont(turn_duration, 0.25), 2) AS p25_turn_duration,
    round(quantile_cont(turn_duration, 0.75), 2) AS p75_turn_duration,
    round(quantile_cont(turn_duration, 0.90), 2) AS p90_turn_duration,
    round(quantile_cont(turn_duration, 0.95), 2) AS p95_turn_duration,
    round(quantile_cont(turn_duration, 0.99), 2) AS p99_turn_duration,
    round(max(turn_duration), 2) AS max_turn_duration

FROM turns

GROUP BY
    player,
    model,
    depth,
    iterations

ORDER BY
    player,
    model,
    depth,
    iterations
""".trimIndent(),
)

In [8]:
duration_stats.groupBy { model and depth and iterations}.aggregate { mean() }.flatten()

model,depth,iterations,depth1,iterations1,turns,total_duration,mean_turn_duration,median_turn_duration,stddev_turn_duration,min_turn_duration,p25_turn_duration,p75_turn_duration,p90_turn_duration,p95_turn_duration,p99_turn_duration,max_turn_duration
MCTS,1,0,1.000000,0.000000,91000.000000,2476.500000,0.090000,0.000000,2.570000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,79.000000
MCTS,1,999,1.000000,999.000000,91000.000000,19286055.000000,574.260000,563.000000,383.295000,2.500000,235.500000,866.250000,993.500000,1084.675000,1596.180000,3579.500000
MINIMAX,1,0,1.000000,0.000000,91000.000000,630540.000000,21.435000,3.000000,82.130000,0.000000,1.000000,8.000000,31.000000,95.500000,394.110000,3342.500000
MINIMAX,3,0,3.000000,0.000000,91000.000000,54033886.500000,1679.865000,164.000000,7692.775000,0.000000,42.500000,983.500000,3470.100000,6877.500000,23892.925000,518746.500000


In [19]:
duration_stats.groupBy { model and depth and iterations}.aggregate { mean() }.flatten().convert { depth and iterations }.with { it.toString() }
//	.schema()
	.select { model and depth and iterations and nameContains("duration") }
	.update { colsOf<Number>() }.with {
	ln1p(it.toDouble())
}

model,depth,iterations,total_duration,mean_turn_duration,median_turn_duration,stddev_turn_duration,min_turn_duration,p25_turn_duration,p75_turn_duration,p90_turn_duration,p95_turn_duration,p99_turn_duration,max_turn_duration
MCTS,1,0,7.815005,0.086178,0.000000,1.272566,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.382027
MCTS,1,999,16.774893,6.354822,6.335054,5.951410,1.252763,5.465948,6.765327,6.902240,6.989957,7.375995,8.183258
MINIMAX,1,0,13.354333,3.110622,1.386294,4.420406,0.000000,0.693147,2.197225,3.465736,4.569543,5.979164,8.114773
MINIMAX,3,0,17.805122,7.427064,5.105945,8.948167,0.000000,3.772761,6.892134,8.152227,8.836156,10.081380,13.159173
